In [15]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas
import os

device = "mps"
print(torch.backends.mps.is_available())

True


In [18]:
ds = load_dataset("cais/mmlu", "all", split="test").shuffle(seed=100).select(range(50))
row = ds[0]
print(row["question"])
print(row["choices"])
print(row["answer"])

Solve for the sum of all possible values of $x$ when $3^{x^2+4x+4}=9^{x+2}$.
['1', '2', '-2', '0']
2


In [4]:
model_name = "allenai/OLMo-2-0425-1B"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16).to(device)
model.eval()

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

Olmo2ForCausalLM(
  (model): Olmo2Model(
    (embed_tokens): Embedding(100352, 2048, padding_idx=100277)
    (layers): ModuleList(
      (0-15): 16 x Olmo2DecoderLayer(
        (self_attn): Olmo2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Olmo2RMSNorm((2048,), eps=1e-06)
          (k_norm): Olmo2RMSNorm((2048,), eps=1e-06)
        )
        (mlp): Olmo2MLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (post_attention_layernorm): Olmo2RMSNorm((2048,), eps=1e-06

In [19]:
letters = ["A", "B", "C", "D"]
letter_ids = []
for letter in letters:
    ids = tok.encode(" " + letter, add_special_tokens=False)
    assert len(ids) == 1
    letter_ids.append(ids[0])

def format_prompt(row):
    text = row["question"] + "\n"
    for letter, choice in zip(letters, row["choices"]):
        text += f"{letter}. {choice}\n"
    text += "Answer:"
    return text

prompt = format_prompt(row)
print(prompt)

Solve for the sum of all possible values of $x$ when $3^{x^2+4x+4}=9^{x+2}$.
A. 1
B. 2
C. -2
D. 0
Answer:


In [20]:
def get_choice_probs(prompt, letter_ids):


    inputs = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])

    next_token_scores = out.logits[0, -1]
    letter_scores = next_token_scores[letter_ids].float()
    probs = torch.softmax(letter_scores, dim=-1)
    all_probs = torch.softmax(next_token_scores.float(), dim=-1)
    letter_mass = all_probs[letter_ids].sum().item()
    return probs.tolist(), letter_mass


In [23]:
results = []
for row in ds:
    probs, mass = get_choice_probs(format_prompt(row), letter_ids)
    pred = probs.index(max(probs))
    answer = ["A", "B", "C", "D"][int(row["answer"])]
    results.append({
        "subject": row["subject"],
        "answer": answer,
        "p_A": probs[0],
        "p_B": probs[1],
        "p_C": probs[2],
        "p_D": probs[3],
        "pred": pred,
        "confidence": max(probs),
        "is_correct": int(row["answer"]) == pred,
        "letter_mass": mass,
    })

In [26]:
df = pandas.DataFrame(results)
os.makedirs("../results", exist_ok=True)
df.to_csv("../results/olmo2-1b-base_sample50.csv", index=False)
print(f"Accuracy:         {df['is_correct'].mean():.3f}")
print(f"Mean confidence:  {df['confidence'].mean():.3f}")
print(f"Mean letter mass: {df['letter_mass'].mean():.3f}")


Accuracy:         0.420
Mean confidence:  0.364
Mean letter mass: 0.849
